# Satellite Telemetry Data Anomaly Detection

This notebook builds a machine learning pipeline to detect anomalies in **simulated** satellite telemetry using **Pandas**, **Scikit-learn**, and **Matplotlib**.

**Workflow:** generate data → preprocess → feature engineering → model tuning → evaluate & visualize.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.telemetry_simulator import SENSOR_COLUMNS, generate_telemetry

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
RANDOM_STATE = 42

## 1. Load simulated telemetry

In [ ]:
raw = generate_telemetry(n_samples=6000, anomaly_rate=0.045, seed=RANDOM_STATE)
print(f"Samples: {len(raw):,}  |  Anomalies: {raw['is_anomaly'].sum()} ({raw['is_anomaly'].mean():.2%})")
raw.head()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.ravel()
for ax, col in zip(axes, SENSOR_COLUMNS):
    ax.plot(raw["timestamp_idx"], raw[col], linewidth=0.6, alpha=0.85)
    ax.scatter(
        raw.loc[raw["is_anomaly"] == 1, "timestamp_idx"],
        raw.loc[raw["is_anomaly"] == 1, col],
        s=8,
        c="crimson",
        label="anomaly",
        zorder=3,
    )
    ax.set_title(col.replace("_", " "))
    ax.legend(loc="upper right", fontsize=7)
plt.suptitle("Raw telemetry with labeled anomalies", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Preprocessing (Pandas)

In [ ]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Clean, sort, and clip sensor channels."""
    out = df.sort_values("timestamp_idx").reset_index(drop=True).copy()
    out = out.drop_duplicates(subset=["timestamp_idx"])
    for col in SENSOR_COLUMNS:
        out[col] = out[col].interpolate(method="linear").bfill().ffill()
    out["storage_used_pct"] = out["storage_used_pct"].clip(0, 100)
    return out


df = preprocess(raw)
df.describe().T

## 3. Feature engineering

Rolling statistics, temporal deltas, and physics-inspired ratios improve separability between normal and faulty regimes.

In [ ]:
def engineer_features(df: pd.DataFrame, window: int = 15) -> pd.DataFrame:
    feat = df[["timestamp_idx"] + SENSOR_COLUMNS].copy()
    for col in SENSOR_COLUMNS:
        roll = feat[col].rolling(window, min_periods=3)
        feat[f"{col}_roll_mean"] = roll.mean()
        feat[f"{col}_roll_std"] = roll.std()
        feat[f"{col}_delta"] = feat[col].diff()
        feat[f"{col}_z"] = (feat[col] - feat[f"{col}_roll_mean"]) / (feat[f"{col}_roll_std"] + 1e-6)
    feat["gyro_magnitude"] = np.sqrt(
        feat["gyro_x_dps"] ** 2 + feat["gyro_y_dps"] ** 2 + feat["gyro_z_dps"] ** 2
    )
    feat["power_ratio"] = feat["solar_current_a"] / (feat["battery_voltage_v"] + 1e-6)
    feat["thermal_per_volt"] = feat["cpu_temp_c"] / (feat["battery_voltage_v"] + 1e-6)
    return feat.dropna()


features = engineer_features(df)
labels = df.loc[features.index, "is_anomaly"].values
features = features.reset_index(drop=True)
feature_cols = [c for c in features.columns if c != "timestamp_idx"]
X = features[feature_cols]
y = labels
print(f"Feature matrix: {X.shape}  |  Positive rate: {y.mean():.2%}")

## 4. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 5. Baseline: Isolation Forest (unsupervised)

In [ ]:
iso = IsolationForest(contamination=y.mean(), random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_train_s)
iso_pred = (iso.predict(X_test_s) == -1).astype(int)
print("Isolation Forest")
print(classification_report(y_test, iso_pred, target_names=["normal", "anomaly"]))

## 6. Supervised models with hyperparameter tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_pipeline = Pipeline([
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced")),
])
rf_grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [8, 12, None],
    "clf__min_samples_leaf": [1, 3, 5],
}
rf_search = GridSearchCV(
    rf_pipeline, rf_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
rf_search.fit(X_train_s, y_train)
print("Best Random Forest params:", rf_search.best_params_)
print(f"CV F1: {rf_search.best_score_:.4f}")

In [ ]:
gb_pipeline = Pipeline([
    ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])
gb_grid = {
    "clf__n_estimators": [80, 120],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [3, 5],
}
gb_search = GridSearchCV(
    gb_pipeline, gb_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
gb_search.fit(X_train_s, y_train)
print("Best Gradient Boosting params:", gb_search.best_params_)
print(f"CV F1: {gb_search.best_score_:.4f}")

In [ ]:
models = {
    "Isolation Forest": (iso, None),
    "Random Forest (tuned)": (rf_search.best_estimator_, True),
    "Gradient Boosting (tuned)": (gb_search.best_estimator_, True),
}

results = []
for name, (model, has_proba) in models.items():
    if has_proba:
        pred = model.predict(X_test_s)
        proba = model.predict_proba(X_test_s)[:, 1]
    else:
        pred = (model.predict(X_test_s) == -1).astype(int)
        scores = -model.score_samples(X_test_s)
        proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    results.append({
        "model": name,
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "pred": pred,
        "proba": proba,
    })

metrics_df = pd.DataFrame(results)[["model", "f1", "roc_auc"]]
metrics_df.sort_values("f1", ascending=False)

## 7. Visualize performance metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x = np.arange(len(metrics_df))
width = 0.35
axes[0].bar(x - width / 2, metrics_df["f1"], width, label="F1", color="steelblue")
axes[0].bar(x + width / 2, metrics_df["roc_auc"], width, label="ROC-AUC", color="darkorange")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_df["model"], rotation=15, ha="right")
axes[0].set_ylabel("Score")
axes[0].set_title("Model comparison on hold-out set")
axes[0].legend()
axes[0].set_ylim(0, 1.05)

best = max(results, key=lambda r: r["f1"])
print(f"Best model by F1: {best['model']}")
print(classification_report(y_test, best["pred"], target_names=["normal", "anomaly"]))

ConfusionMatrixDisplay(confusion_matrix(y_test, best["pred"]), display_labels=["normal", "anomaly"]).plot(
    ax=axes[1], cmap="Blues", colorbar=False
)
axes[1].set_title(f"Confusion matrix — {best['model']}")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for r in results:
    RocCurveDisplay.from_predictions(y_test, r["proba"], name=r["model"], ax=ax)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_title("ROC curves")
plt.tight_layout()
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, best["proba"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, linewidth=2, label=best["model"])
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall curve (best model)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Anomaly overlay on telemetry timeline

In [ ]:
# Align test indices with original timeline for plotting
test_idx = X_test.index
plot_df = df.loc[test_idx].copy()
plot_df["pred_anomaly"] = best["pred"]
plot_df["true_anomaly"] = y_test

sensor = "battery_voltage_v"
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(plot_df["timestamp_idx"], plot_df[sensor], color="gray", linewidth=0.7, label=sensor)
ax.scatter(
    plot_df.loc[plot_df["true_anomaly"] == 1, "timestamp_idx"],
    plot_df.loc[plot_df["true_anomaly"] == 1, sensor],
    s=25,
    facecolors="none",
    edgecolors="green",
    linewidths=1.2,
    label="true anomaly",
)
ax.scatter(
    plot_df.loc[plot_df["pred_anomaly"] == 1, "timestamp_idx"],
    plot_df.loc[plot_df["pred_anomaly"] == 1, sensor],
    s=12,
    c="crimson",
    alpha=0.8,
    label="predicted anomaly",
)
ax.set_xlabel("Time index")
ax.set_ylabel(sensor)
ax.set_title(f"Anomaly detection overlay — {best['model']}")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
if hasattr(rf_search.best_estimator_.named_steps["clf"], "feature_importances_"):
    imp = rf_search.best_estimator_.named_steps["clf"].feature_importances_
    top = pd.Series(imp, index=feature_cols).sort_values(ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(9, 5))
    top.sort_values().plot(kind="barh", ax=ax, color="teal")
    ax.set_title("Top feature importances (tuned Random Forest)")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()

## 9. Export dataset (optional)

In [ ]:
out_path = PROJECT_ROOT / "data" / "telemetry_simulated.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")